In [2]:
%pip install altair vega_datasets
%pip install plotly pandas 

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# import libraries
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import altair as alt
import numpy as np
from vega_datasets import data
import plotly.express as px


In [4]:
# import hospital csv 
hospital = pd.read_csv("Hospital_Provider_Cost_Report_2023_updated.csv", 
                 encoding='latin-1')

print(hospital.shape)

(6103, 117)


In [5]:
state_counts = hospital["State Code"].value_counts().sort_index() 

# Update 'state' to match the actual column name in your dataset
state_col = "State Code"

# Check which US states are missing
all_states = [
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA",
    "HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
    "MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
    "NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
    "SD","TN","TX","UT","VT","VA","WA","WV","WI","WY",
]

print(f"Total unique states in data: {hospital[state_col].nunique()}")
print(f"Total hospitals: {len(hospital)}\n")

# Check for states in data that aren't in all_states (e.g. territories)
extra = [s for s in state_counts.index if s not in all_states]
if extra:
    print(f"\nStates/territories in data but not in US states list:")
    for s in extra:
        print(f"{s:<10} {state_counts[s]:<10}")

Total unique states in data: 55
Total hospitals: 6103


States/territories in data but not in US states list:
DC         12        
GU         2         
MP         1         
PR         64        
VI         2         


In [6]:
# PRE-PROCESSING THE DATA TO GET THE MAIN DATASET 

# Rows to get rid of 
# "State Code" - PR, DC, VI, GU, MP

# Columns I want to keep 

keep_columns = ["Hospital Name",
                "City",
                "State Code",
                "Zip Code",
                "County",
                "Rural Versus Urban",
                "CCN Facility Type",
                "Provider Type",
                "Type of Control",
                "Fiscal Year Begin Date",
                "Fiscal Year End Date",
                "FTE - Employees on Payroll",
                "Total Days (V + XVIII + XIX + Unknown)",
                "Number of Beds",
                "Total Bed Days Available",
                "Total Discharges (V + XVIII + XIX + Unknown)",
                "Number of Beds + Total for all Subproviders",
                "Total Bad Debt Expense",
                "Total Unreimbursed and Uncompensated Care",
                "Total Costs",
                "Combined Outpatient + Inpatient Total Charges",
                "Less Total Operating Expense",
                "Cost To Charge Ratio",
                "Net Patient Revenue",
                "Net Income from Service to Patients",
                "Net Income"
]

state_dict = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire',
    'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina',
    'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania',
    'RI': 'Rhode Island', 'SC': 'South Carolina', 'SD': 'South Dakota', 'TN': 'Tennessee',
    'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington',
    'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}

remove_territories = ["PR", "DC", "VI", "GU", "MP"]

hospital_filter = hospital[~(hospital["State Code"]).isin(remove_territories)][keep_columns] # filter dataset for the states I want 
hospital_filter = hospital_filter.dropna(thresh=len(hospital_filter.columns) * 0.5) # drop rows with more than 50% missing values
hospital_filter["Staff Per Bed"] = hospital_filter["FTE - Employees on Payroll"] / hospital_filter["Number of Beds + Total for all Subproviders"] # new variable 
hospital_filter["Clinical Operating Margin"] = hospital_filter["Net Income from Service to Patients"] / hospital_filter["Net Patient Revenue"] # new variable 
hospital_filter["Net Revenue Per Bed"] = hospital_filter["Net Patient Revenue"] / hospital_filter["Number of Beds + Total for all Subproviders"]  # new variable 
hospital_filter['State Name'] = hospital_filter['State Code'].map(state_dict) # new varaible - state name 
hospital_filter["Operating Expense Ratio"] = hospital_filter["Less Total Operating Expense"] / hospital_filter["Net Patient Revenue"] # new variable 

# make bins for the type of control vs. clinical operating margin visualization 
bins = [0, 75, 250, 10000] 
labels = ['Small', 'Medium', 'Large']
hospital_filter['Size Category'] = pd.cut(hospital_filter['Number of Beds + Total for all Subproviders'], bins=bins, labels=labels, right=False)

hospital_filter.to_csv("Cleaned_DF_Hospital_Provider_Cost_Report_2023.csv", index=False)

In [7]:
## Getting the dataframe that will be used for D3 visualization 
scatterplot = hospital_filter[['Type of Control', 'Number of Beds + Total for all Subproviders', 'Cost To Charge Ratio', 'Rural Versus Urban', "Operating Expense Ratio"]].dropna().query("0 < `Operating Expense Ratio` <= 3 and `Number of Beds + Total for all Subproviders` < 2500")
json_data = scatterplot.to_json('scatterplot_json', orient='records')


In [15]:
## there are impossible outliers becuase the margin should only be between 1 and -1, which might be a data error 

# Small: < 75 beds (This is your 50th percentile/median).
# Medium: 75–250 beds (Captures the bulk of the 3rd and part of the 4th quartile).
# Large: 250+ beds (Captures the top 15-20% of hospitals).

#  1 = Voluntary Non‐Profit‐
# Church, 2 = Voluntary Non‐Profit‐Other, 3 = Proprietary‐
# Individual, 4 = Proprietary‐Corporation, 5 = Proprietary‐
# Partnership, 6 = Proprietary‐Other, 7 = Governmental‐
# Federal, 8 = Governmental‐City‐County, 9 = Governmental‐
# County, 10 = Governmental‐State, 11 = Governmental‐
# Hospital District, 12 = Governmental‐City, 13 = Governmental‐
# Other

control_mapping = {
    1: "Voluntary Non-Profit-Church",
    2: "Voluntary Non-Profit-Other",
    3: "Proprietary-Individual",
    4: "Proprietary-Corporation", 
    5: "Proprietary-Partnership",
    6: "Proprietary-Other",
    7: "Governmental-Federal",
    8: "Government-City-County",
    9: "Governmental County",
    10: "Governmental-State",
    11: "Governmental-Hospital District",
    12: "Governmental-City",
    13: "Governmental-Other"
}

hospital_filter = hospital_filter.dropna(subset=['Size Category'])
hospital_filter['Type of Control'] = hospital_filter['Type of Control'].replace(control_mapping)


In [16]:
# 1. Define the selection. 
# We add toggle=False so it acts like a "one-at-a-time" selector.
size_options = ['Small', 'Medium', 'Large']
radio_button = alt.binding_radio(options=size_options, name="Hospital Size: ")
size_param = alt.param(
    value='All',                        # ← defaults to All
    name='size_sel',                    
    bind=alt.binding_radio(
        options=['All', 'Small', 'Medium', 'Large'],  # ← add All
        name='Select Hospital Size: '
    )
)

# 2. Build the chart
boxplots = alt.Chart(hospital_filter).transform_filter(
    # Your existing filter for the margin bounds
    "datum['Clinical Operating Margin'] > -1 && datum['Clinical Operating Margin'] < 1"
).transform_filter(
    "size_sel == null || size_sel == 'All' || datum['Size Category'] == size_sel"  # ← add null check
).mark_boxplot(extent='min-max', size=10, ticks=True, opacity=1).encode(
    y=alt.Y('Type of Control:O', scale=alt.Scale(paddingInner=0.5), title='Ownership Type'),
    x=alt.X('Clinical Operating Margin:Q', scale=alt.Scale(domain=[-1, 1])),
    color=alt.Color('Size Category:N', 
                    scale=alt.Scale(
                    domain=['Small', 'Medium', 'Large'],  # ← pin order
                    range=["#6ad5ff", '#ff892a', "#A44700"]  # ← locked to domain
                ),
                    legend=alt.Legend(title='Hospital Size')),
    yOffset=alt.YOffset('Size Category:N', scale=alt.Scale(paddingInner=0.3))
).properties(
    width=500,
    height=800, 
    title = "Boxplot of Clinical Operating Margin by Ownership Type and Size"
)

# 3. Define the zero line
zero_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(
    color='black', strokeWidth=2, strokeDash=[4, 4] 
).encode(x='x:Q')

# 4. Combine and add the params
final_chart = (boxplots + zero_line).add_params(
    size_param
)

final_chart.save('boxplot_with_radio.html')

In [12]:

# 1. Drop rows where FIPS is missing
hospital_with_fips = hospital_with_fips.dropna(subset=['fips'])
hospital_with_fips = hospital_with_fips.dropna(subset=['Net Revenue Per Bed'])


# 2. Now convert to integer (ensures it matches the map's ID format)
hospital_with_fips['fips'] = hospital_with_fips['fips'].astype(int)# 1. Load the background geometry
states_geo = alt.topo_feature(data.us_10m.url, 'states')
hospital_with_fips['fips_state'] = hospital_with_fips['fips'].apply(lambda x: int(str(int(x)).zfill(5)[:2]))

# both county and state
state_map = hospital_with_fips.groupby(['fips_state', 'State Name'], as_index=False)['Net Revenue Per Bed'].mean()
county_map = hospital_with_fips.groupby(['fips', 'County'], as_index=False)['Net Revenue Per Bed'].mean()


# 2. Create the Choropleth Map
# We use .transform_lookup to "bridge" your dataframe and the map geometry
map = alt.Chart(states_geo).mark_geoshape(
    stroke='white',
    strokeWidth=0.5).project(
    type='albersUsa'
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(state_map, 'fips_state', ['Net Revenue Per Bed', 'State Name'])
).encode(
    # Use the name created in the 'as' part of the aggregate below
    color=alt.Color('Net Revenue Per Bed:Q', 
        scale=alt.Scale(scheme='oranges'),
        title='Avg Revenue per Bed ($)'
    ),
    tooltip=[
        alt.Tooltip('State Name:N'),
        alt.Tooltip('Net Revenue Per Bed:Q', format='$,.0f', title='Average Net Revenue per Bed')
    ]
).properties(
    width=650,
    height=500,
    title='Average Net Patient Revenue per Bed by State (2023)'
)
hospital_with_fips

,Hospital Name,City,State Code,Zip Code,County,Rural Versus Urban,CCN Facility Type,Provider Type,Type of Control,Fiscal Year Begin Date,...,Clinical Operating Margin,Net Revenue Per Bed,State Name,Operating Expense Ratio,Size Category,county_name,state_abbr,fips,region_name,fips_state
0,IRWIN COUNTY HOSPITAL,OCILLA,GA,31774,IRWIN,R,STH,1,Governmental County,2022-12-01,...,-0.947402,2.434047e+04,Georgia,1.947402,Small,IRWIN,GA,13155,South,13
1,LAKE BEHAVIORAL HOSPITAL,WAUKEGAN,IL,60085,LAKE,U,PH,4,Proprietary-Partnership,2022-10-01,...,0.122490,6.077204e+04,Illinois,0.877510,Medium,LAKE,IL,17097,Midwest,17
2,EVEREST REHABILITATION HOSPITAL BENT,ROGERS,AR,72758-1347,BENTON,U,RH,5,Proprietary-Other,2023-01-01,...,-0.687706,4.029036e+04,Arkansas,1.687706,Small,BENTON,AR,5007,South,5
3,OCEANS BEHAVIORAL HOSPITAL CORPUS CH,CORPUS CHRISTI,TX,78404,NUECES,U,PH,4,Proprietary-Corporation,2022-11-17,...,-1.730492,3.004193e+04,Texas,2.730492,Small,NUECES,TX,48355,South,48
4,WILLIAMSBURG REGIONAL HOSPITAL,KINGSTREE,SC,29556,WILLIAMSBURG,R,CAH,1,Voluntary Non-Profit-Other,2022-10-01,...,-0.599921,2.169195e+05,South Carolina,1.599921,Small,WILLIAMSBURG,SC,45089,South,45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5915,CAMERON MEMORIAL COMMUNITY HOSPITAL,ANGOLA,IN,47803-,STEUBEN,R,CAH,1,Voluntary Non-Profit-Other,2022-10-01,...,-0.014348,3.897171e+06,Indiana,1.014348,Small,STEUBEN,IN,18151,Midwest,18
5917,BMH - CALHOUN,CALHOUN,MS,38916,CALHOUN,R,CAH,1,Voluntary Non-Profit-Church,2022-10-01,...,-0.018651,1.608226e+05,Mississippi,1.018651,Medium,CALHOUN,MS,28013,South,28
5918,TEXAS HEALTH PRESBYTERIAN HOSPITAL P,PLANO,TX,75093,COLLIN,U,STH,1,Voluntary Non-Profit-Other,2023-01-01,...,0.174463,1.541495e+06,Texas,0.825537,Large,COLLIN,TX,48085,South,48
5919,UNIVERSITY OF VERMONT MEDICAL CENTER,BURLINGTON,VT,05401,CHITTENDEN,R,STH,1,Voluntary Non-Profit-Other,2022-10-01,...,-0.113028,3.952307e+06,Vermont,1.113028,Large,CHITTENDEN,VT,50007,Northeast,50


In [13]:
import altair as alt

# 1. Define selection
click_state = alt.selection_point(fields=['fips_state'], name='state_selector')

# 2. THE MAP
map_chart = alt.Chart(states_geo).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).project(
    type='albersUsa'
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(state_map, 'fips_state', ['Net Revenue Per Bed', 'State Name', 'fips_state'])
).encode(
    color=alt.Color('Net Revenue Per Bed:Q', 
        scale=alt.Scale(scheme='oranges'),
        title='Avg Revenue/Bed ($)',
        # legend=alt.Legend(orient='right') # You can also try 'left' or 'bottom'
    ),
    opacity=alt.condition(click_state, alt.value(1), alt.value(0.2)),
    tooltip=[
        alt.Tooltip('State Name:N'),
        alt.Tooltip('Net Revenue Per Bed:Q', format='$,.0f')
    ]
).add_params(
    click_state
).properties(
    width=500,  # Keeping the map large
    height=400,
    title='Choropleth Map: Average Net Patient Revenue per Bed by State (2023)'
)

# 3. THE HISTOGRAM (Slightly Smaller)
hist_chart = alt.Chart(hospital_with_fips).mark_bar().encode(
    x=alt.X('Net Revenue Per Bed:Q', 
            bin=alt.Bin(maxbins=20), # Reduced bins for a cleaner small look
            title='Net Revenue per Bed ($)'),
    y=alt.Y('count()', title='Number of Hospitals'),
    color=alt.value('#d95f02')
).transform_filter(
    click_state
).properties(
    width=300,  # Reduced from 400
    height=300, # Reduced from 400
    title='Hospital Revenue Density for Selected State'
)

# 4. Combine and Resolve Legend
final_viz = (map_chart | hist_chart).resolve_legend(
    color='independent' # This keeps the color legend attached to the map specifically
).configure_view(
    strokeWidth=0
).configure_legend(
    orient='right', # Ensures it stays to the right of the MAP
    padding=10
).configure(
    # [left, top, right, bottom] padding in pixels
    padding={'left': 10, 'top': 10, 'right': 50, 'bottom': 10} 
)

final_viz.save('choropleth_with_histogram.html')